My flow reads a synthetic practice-record CSV, groups by domain, flags scores below a named cutoff, and a person reviews before anything happens.

# GapLift-style practice-data capstone flow

**Safety:** The records in this exercise are synthetic. A low observed practice score is only a review signal. Recommendations require human review before any external action or change. This notebook performs no external action, sends no message, changes no website content, and reads no personal learner data.

**This is not a predictive model.** It does not estimate exam outcomes, certification scores, or learner ability. The configurable threshold below is a simple teaching rule, not the website's recommendation algorithm.

## Data and reproducibility

The small eight-row CSV is a deterministic teaching fixture. It contains six valid scores, one nonnumeric score, and one blank score. Both invalid rows are deliberately present so the audit trail is visible. If a requested file is missing, the loader raises by default. The missing-file test explicitly opts into a labeled mock fallback; the output metadata identifies the fallback and why it was used.

In [1]:
from pathlib import Path
import subprocess
import sys

candidate_dirs = [Path.cwd(), Path.cwd() / "capstone"]
CAPSTONE_DIR = next((p for p in candidate_dirs if (p / "capstone_analysis.py").is_file()), None)
if CAPSTONE_DIR is None:
    raise FileNotFoundError("Place capstone_analysis.py beside this notebook, or in a capstone/ folder.")
sys.path.insert(0, str(CAPSTONE_DIR))
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(CAPSTONE_DIR / "requirements.txt")],
    check=False,
    capture_output=True,
    text=True,
)
if install.returncode:
    raise RuntimeError(f"Runtime setup failed:\n{install.stdout}\n{install.stderr}")
print("Pinned runtime requirements are installed.")

import pandas as pd
from capstone_analysis import (
    DEFAULT_THRESHOLD,
    analyze_records,
    build_review_graph,
    load_data,
    run_course_checks,
    validate_required_columns,
)

FIXTURE_PATH = CAPSTONE_DIR / "fixtures" / "practice_records.csv"
print("Python environment ready.")
print("Fixture:", FIXTURE_PATH)
print("Threshold:", DEFAULT_THRESHOLD, "(scores strictly below this value are flagged)")

Pinned runtime requirements are installed.


Python environment ready.
Fixture: /workspace/comptia-a-plus-review-capstone/capstone/fixtures/practice_records.csv
Threshold: 70.0 (scores strictly below this value are flagged)


## 1. Load the requested CSV

In [2]:
# No fallback is enabled for this primary run; a missing path fails visibly.
raw, source_metadata = load_data(FIXTURE_PATH, use_mock_fallback=False)
print(source_metadata["notice"])
print("Source kind:", source_metadata["source_kind"])
print("Fallback used:", source_metadata["fallback_used"])

Loaded the supplied CSV file.
Source kind: csv_file
Fallback used: False


## 2. Inspect before changing values

In [3]:
display(raw.head())
print("shape:", raw.shape)
print("info:")
raw.info()
print("describe (raw CSV columns):")
display(raw.describe(include="all"))

,record_id,domain,practice_score
0,r001,compute,88
1,r002,storage,72
2,r003,storage,61
3,r004,network,93
4,r005,compute,not-a-score


shape: (8, 3)
info:
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   record_id       8 non-null      str   
 1   domain          8 non-null      str   
 2   practice_score  8 non-null      string
dtypes: str(2), string(1)
memory usage: 324.0 bytes
describe (raw CSV columns):


,record_id,domain,practice_score
count,8,8,8
unique,8,4,8
top,r001,compute,88
freq,1,2,1


## 3. Validate the required columns

The required fields are `record_id`, `domain`, and `practice_score`. A missing field raises a readable error instead of quietly proceeding.

In [4]:
validate_required_columns(raw)
print("Required columns present:", ["record_id", "domain", "practice_score"])

Required columns present: ['record_id', 'domain', 'practice_score']


## 4. Clean scores and keep an audit trail

The analysis module applies `pd.to_numeric(raw_score, errors="coerce")`. Invalid or blank scores become missing, are excluded from means and flags, and are listed with their row ID and exclusion reason. This does not delete or modify the input CSV.

In [5]:
# Show the exact conversion before summarizing. The analysis helper repeats this
# conversion as part of its reusable load → validate → clean flow.
numeric_preview = pd.to_numeric(raw["practice_score"], errors="coerce")
print("Numeric conversion preview (NaN means the score is invalid or blank):")
display(pd.DataFrame({"record_id": raw["record_id"], "raw_score": raw["practice_score"], "numeric_score": numeric_preview}))

threshold = 70.0
result = analyze_records(raw, threshold=threshold, source_metadata=source_metadata)
print("original rows:", result["metadata"]["original_count"])
print("retained rows:", result["metadata"]["retained_count"])
print("dropped rows:", result["dropped_count"])
print("cleaning policy:", result["metadata"]["invalid_score_policy"])
print("excluded-row audit:")
display(pd.DataFrame(result["audit_rows"]))

Numeric conversion preview (NaN means the score is invalid or blank):


,record_id,raw_score,numeric_score
0,r001,88,88.0
1,r002,72,72.0
2,r003,61,61.0
3,r004,93,93.0
4,r005,not-a-score,<NA>
5,r006,58,58.0
6,r007,81,81.0
7,r008,,<NA>


original rows: 8
retained rows: 6
dropped rows: 2
cleaning policy: pd.to_numeric(errors='coerce'); exclude invalid or blank scores from summaries and flags; preserve each exclusion and reason in audit_rows.
excluded-row audit:


,record_id,domain,raw_practice_score,excluded,reason
0,r005,compute,not-a-score,True,practice_score is not numeric; coerced to missing
1,r008,network,,True,blank practice_score


## 5. Summarize by domain

In [6]:
print("Mean practice score by domain (valid numeric rows only):")
display(pd.DataFrame(result["summary"]))

Mean practice score by domain (valid numeric rows only):


,domain,mean_practice_score,record_count
0,compute,88.0,1
1,database,69.5,2
2,network,93.0,1
3,storage,66.5,2


## 6. Flag a review signal

The named threshold is `threshold = 70.0`; only scores **strictly below** 70 are flagged. This is intentionally a low-score rule for review. It differs from the course's example that describes values above a threshold: the direction here reflects the chosen question (“which observed practice scores merit another look?”), not a changed course requirement. Exactly 70 is not flagged.

In [7]:
print("threshold:", result["metadata"]["threshold"])
print(result["metadata"]["signal_note"])
print("flagged rows:")
display(pd.DataFrame(result["flagged"]))

# The reusable output is a plain dictionary containing summary, flagged, and dropped_count.
assert set(["summary", "flagged", "dropped_count"]).issubset(result)
print("returned keys:", sorted(result.keys()))

threshold: 70.0
A score below the configurable cutoff is a review signal, not a diagnosis, prediction, or learner outcome.
flagged rows:


,record_id,domain,practice_score
0,r003,storage,61.0
1,r006,database,58.0


returned keys: ['audit_rows', 'dropped_count', 'flagged', 'metadata', 'summary']


## 7. Four requested checks

These checks assert the course's valid-fixture result, explicitly labeled missing-file fallback, missing-column error, and a threshold with no flags. The four `ok` messages are printed only after each check's assertions pass; `all tests passed` is printed only when all four have passed.

In [8]:
run_course_checks(FIXTURE_PATH, CAPSTONE_DIR / "fixtures" / "missing_column.csv")

ok 1 — valid fixture summary, flags, and cleaning counts
fallback notice: DEMO FALLBACK: using explicitly labeled synthetic records; these are not learner outcomes.
fallback reason: Input CSV not found: /workspace/comptia-a-plus-review-capstone/capstone/fixtures/intentionally_missing.csv
ok 2 — missing-file fallback is explicit and visible
ok 3 — missing-column error is clear
ok 4 — lower boundary threshold produces no flags
all tests passed


### Additional cleaning edge cases

An empty file with the expected columns and a file where every score is invalid are reported distinctly. Neither is mislabeled as an ordinary “no flags” result.

In [9]:
all_invalid = analyze_records([
    {"record_id": "bad-1", "domain": "compute", "practice_score": "unknown"},
    {"record_id": "bad-2", "domain": "network", "practice_score": ""},
])
empty = analyze_records([])
print("all invalid:", all_invalid["metadata"]["clean_status"], "dropped:", all_invalid["dropped_count"])
print("empty input:", empty["metadata"]["clean_status"], "original:", empty["metadata"]["original_count"])

all invalid: no_valid_scores dropped: 2
empty input: empty_input original: 0


## 8. Real LangGraph review pause and resume

The graph has `load`, `clean`, `analyze`, and `human_review` nodes. It uses serializable dictionary/list/scalar state and an in-memory checkpointer. The reviewer interrupt occurs after analysis and before the workflow returns a decision. There is no external-action node.

In [10]:
graph = build_review_graph()
approve_config = {"configurable": {"thread_id": "notebook-demo-approve"}}
paused = graph.invoke(
    {
        "data_path": str(FIXTURE_PATH),
        "threshold": threshold,
        "use_mock_fallback": False,
    },
    approve_config,
)
assert "__interrupt__" in paused
review_packet = paused["__interrupt__"][0].value
print("PAUSED before reviewer decision")
display(review_packet)
print("checkpoint next node:", graph.get_state(approve_config).next)

PAUSED before reviewer decision


{'summary': [{'domain': 'compute',
   'mean_practice_score': 88.0,
   'record_count': 1},
  {'domain': 'database', 'mean_practice_score': 69.5, 'record_count': 2},
  {'domain': 'network', 'mean_practice_score': 93.0, 'record_count': 1},
  {'domain': 'storage', 'mean_practice_score': 66.5, 'record_count': 2}],
 'flagged': [{'record_id': 'r003',
   'domain': 'storage',
   'practice_score': 61.0},
  {'record_id': 'r006', 'domain': 'database', 'practice_score': 58.0}],
 'audit_rows': [{'record_id': 'r005',
   'domain': 'compute',
   'raw_practice_score': 'not-a-score',
   'excluded': True,
   'reason': 'practice_score is not numeric; coerced to missing'},
  {'record_id': 'r008',
   'domain': 'network',
   'raw_practice_score': '',
   'excluded': True,
   'reason': 'blank practice_score'}],
 'safety': 'Review only. No message, site change, or other external action is performed by this workflow.',
 'required_response': {'decision': 'approve or reject',
  'notes': 'reviewer explanation'}}

checkpoint next node: ('human_review',)


### Validation/demo only: scripted decisions

**Do not use these fixed responses as an operational approval.** The next cell exists only so this notebook can be executed reproducibly and show both outcomes. For a real review, stop at the pause above, inspect the packet, obtain a human decision, then resume using `Command(resume={"decision": "approve" or "reject", "notes": "..."})`. Approval here does not perform any external action.

In [11]:
# VALIDATION/DEMO ONLY — illustrative scripted reviewer responses.
# Replace only after a person has inspected the review packet in a real workflow.
from langgraph.types import Command

approved = graph.invoke(
    Command(resume={"decision": "approve", "notes": "Demo reviewer inspected the synthetic summary."}),
    approve_config,
)
print("approved demo status:", approved["workflow_status"])
print("external action:", approved["external_action"])

reject_config = {"configurable": {"thread_id": "notebook-demo-reject"}}
reject_paused = graph.invoke(
    {"data_path": str(FIXTURE_PATH), "threshold": threshold, "use_mock_fallback": False},
    reject_config,
)
assert "__interrupt__" in reject_paused
rejected = graph.invoke(
    Command(resume={"decision": "reject", "notes": "Demo reviewer asks for follow-up."}),
    reject_config,
)
print("rejected demo status:", rejected["workflow_status"])
print("external action:", rejected["external_action"])

approved demo status: review_approved_no_external_action
external action: none
rejected demo status: review_rejected_no_external_action
external action: none


## What this result means

The two flagged rows are only records below the example threshold. They are not evidence of learner weakness, future performance, a certification score, or a prediction. GapLift's website uses saved timed-test outcomes, a recent-answer window, and minimum-answer rules; this notebook's single cutoff is a smaller teaching example and is not equivalent. Human review is required before any external action or change.